### Chapter 4.1 - Softmax Regression

Softmax regression is linear regression's classification counterpart. It predicts a probability distribution over classes instead of one numerical value.


In [ ]:
import math
import random
import numpy as np
import torch


### 4.1.1 Classification

#### 1. Intuition

Classification means predicting a category. A category is a named group, such as `cat`, `dog`, or `shirt`.

A class is one possible category. A label is the correct class for one example. A score, also called a logit, is a raw number a model assigns to a class before turning scores into probabilities.

#### 2. Why this exists

Many ML tasks ask for a decision among choices rather than a number. The model must compare classes, not just output one scalar prediction.


#### 3. Examples

Manual Python: choose the class with the largest score.


In [ ]:
classes = ["cat", "dog", "car"]
scores = [1.2, 3.4, 0.5]
best_index = scores.index(max(scores))
predicted_class = classes[best_index]
predicted_class


PyTorch: compute class scores from features and weights.


In [ ]:
X = torch.tensor([[1.0, 2.0]])
W = torch.tensor([[0.1, 0.2, -0.3], [0.4, -0.5, 0.6]])
b = torch.tensor([0.0, 0.1, -0.1])
logits = X @ W + b
logits


#### 4. Step-by-step breakdown

The manual example compares three class scores.

`max(scores)` finds the largest score.

`scores.index(...)` finds the position of that largest score.

In the tensor example, `X` has one example with two features.

`W` maps two input features to three class scores.

`X @ W + b` returns one score per class.

#### 5. Connection to ML systems

Classification models usually output one score per class. Later, softmax turns those scores into probabilities that sum to 1.

#### 6. Common confusion points

- A class score is not automatically a probability.
- The largest score determines the predicted class.
- The number of output scores should match the number of classes.
- Classification labels are categories, even when represented by integer IDs.


### 4.1.2 Loss Function

#### 1. Intuition

Softmax turns raw class scores into probabilities. A probability is a number between 0 and 1, and class probabilities should sum to 1.

Cross-entropy loss penalizes the model when it assigns low probability to the correct class.

#### 2. Why this exists

Classification training needs a loss that rewards probability placed on the correct class and penalizes probability placed elsewhere.


#### 3. Examples

Manual softmax for three class scores.


In [ ]:
scores = torch.tensor([1.0, 2.0, 0.0])
exp_scores = torch.exp(scores)
probs = exp_scores / exp_scores.sum()
probs


Cross-entropy for the correct class index.


In [ ]:
correct_class = 1
loss = -torch.log(probs[correct_class])
loss


#### 4. Step-by-step breakdown

`torch.exp(scores)` makes all values positive.

Dividing by `exp_scores.sum()` makes the values sum to 1.

`correct_class = 1` means the second class is the true label.

`probs[correct_class]` selects the probability assigned to the true class.

`-torch.log(...)` gives a small loss for high probability and a large loss for low probability.

#### 5. Connection to ML systems

PyTorch usually combines softmax and cross-entropy in `torch.nn.CrossEntropyLoss` for numerical stability. Numerical stability means avoiding avoidable overflow, underflow, or precision problems.

#### 6. Common confusion points

- Softmax probabilities sum to 1 across classes.
- Cross-entropy uses the correct class probability.
- A confident wrong prediction gets a large loss.
- In PyTorch, `CrossEntropyLoss` expects raw logits, not already-softmaxed probabilities.


### 4.1.3 Information Theory Basics

#### 1. Intuition

Information theory studies uncertainty and information.

Surprisal measures how unexpected an event is. A high-probability event is not surprising. A low-probability event is surprising.

The negative logarithm, `-log(p)`, is a common mathematical measure of surprisal.

#### 2. Why this exists

Cross-entropy loss is easier to understand when viewed as surprisal: the model is penalized by how surprised it is when the true class occurs.


#### 3. Examples

Compare surprisal for high and low probabilities.


In [ ]:
high_prob = torch.tensor(0.9)
low_prob = torch.tensor(0.1)
high_surprisal = -torch.log(high_prob)
low_surprisal = -torch.log(low_prob)
high_surprisal, low_surprisal


#### 4. Step-by-step breakdown

`high_prob` represents a confident probability for the event.

`low_prob` represents an unlikely event.

`-log(0.9)` is small because the event is not surprising.

`-log(0.1)` is larger because the event is surprising.

Cross-entropy applies this idea to the true class probability.

#### 5. Connection to ML systems

Classification training minimizes average surprisal of the correct labels under the model's predicted probabilities.

#### 6. Common confusion points

- Information theory terms can sound abstract, but here the key idea is surprise.
- Logarithms turn probability products into sums, which is useful for optimization.
- Cross-entropy is not accuracy; it uses probability confidence.
- A model can have good accuracy but poor cross-entropy if it is badly calibrated.


### 4.1.4 Summary and Discussion

#### 1. Intuition

Softmax regression maps input features to one score per class, converts scores to probabilities, and trains with cross-entropy loss.

#### 2. Why this exists

It is the simplest linear classifier and the foundation for later neural classifiers.


#### 3. Examples

One compact softmax-regression forward pass.


In [ ]:
X = torch.tensor([[1.0, 2.0], [0.5, -1.0]])
W = torch.randn(2, 3)
b = torch.zeros(3)
logits = X @ W + b
probs = torch.softmax(logits, dim=1)
probs.shape, probs.sum(dim=1)


#### 4. Step-by-step breakdown

`X @ W + b` computes raw scores for each example and class.

`torch.softmax(logits, dim=1)` normalizes across class columns.

`probs.shape` is `(2, 3)`: two examples and three classes.

`probs.sum(dim=1)` checks that each example's class probabilities sum to 1.

#### 5. Connection to ML systems

Deep classifiers often end with the same pattern: class logits followed by cross-entropy loss.

#### 6. Common confusion points

- Use `dim=1` when rows are examples and columns are classes.
- Softmax is about relative scores, not absolute score size alone.
- Cross-entropy trains probabilities, not only the final class choice.
- The model is still linear unless hidden layers are added.


### 4.1.5 Exercises

#### 1. Intuition

Exercises should make you inspect logits, probabilities, labels, and loss separately.

#### 2. Why this exists

Classification bugs often come from mixing up class dimension, label format, or whether values are logits or probabilities.


#### 3. Examples

Exercise 1: turn logits into probabilities.


In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.0]])
probs = torch.softmax(logits, dim=1)
probs, probs.sum(dim=1)


Exercise 2: compute cross-entropy with PyTorch from raw logits.


In [ ]:
labels = torch.tensor([0])
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
loss


#### 4. Step-by-step breakdown

Exercise 1 checks softmax normalization across classes.

Exercise 2 checks the PyTorch convention: pass raw logits and integer class labels.

The label `0` means the first class is correct.

#### 5. Connection to ML systems

These exercises prepare for from-scratch and concise softmax-regression implementations.

#### 6. Common confusion points

- Do not pass one-hot labels to `CrossEntropyLoss` in the basic PyTorch use case.
- Do not apply softmax before `CrossEntropyLoss` unless you know the alternative API expects it.
- Check the class dimension before using softmax.
- A probability vector should sum to 1 across classes.
